# Lab 9: Observability, Gateway Tools & Production Debugging
## NovaPay · "Dark Tuesday" — When an Agent Approves What It Shouldn't

**Level path:** 100 (concepts) → 200 (instrumented tools) → 300 (end-to-end tracing through the gateway) → 400 (production operations)

**Duration:** ~90 minutes
**Prerequisites:** Labs 1–8 (concepts from Lab 8's gateway are reused)
**Cost:** ~15 cents. OTEL spans stored in CloudWatch Logs, charged per GB ingested
**Region:** us-west-2

> **This lab provisions real AWS resources**: an IAM role, a Lambda function, a Cognito
> user pool, an AgentCore Gateway with log delivery and tracing enabled. Every resource
> is registered for teardown in section 400.5. Run it.

### What this lab is about

Labs 7 and 8 gave NovaPay's agents memory and a governed front door. Both are live in
production. But production creates a new problem: **when something goes wrong, who knows?**

Last Tuesday at 03:14 UTC, NovaPay's fraud agent approved a $91,500 cross-border transfer
that turned out to be a structured layering attempt. The compliance team spent *14 hours*
reconstructing what happened — reading raw CloudWatch logs line by line, guessing which
tools the agent called, trying to figure out why the risk score was 45 when it should have
been 99.

This lab teaches you to instrument agents so that investigation takes **15 minutes, not
14 hours**. You will set up OpenTelemetry tracing, add custom spans to NovaPay's tools,
wire observability through the AgentCore Gateway, and build the production debugging
workflow that turns "we don't know what happened" into "here is the exact decision tree."

In [ ]:
%pip install -q opentelemetry-api opentelemetry-sdk strands-agents strands-agents-tools \
    bedrock-agentcore mcp "boto3>=1.39.0" requests

In [ ]:
import json, os, time, uuid, textwrap, io, zipfile
import boto3
from datetime import datetime, timezone

# ── Region and model ──
REGION = "us-west-2"
MODEL_ID = "us.anthropic.claude-sonnet-4-20250514-v1:0"
RUN = datetime.now(timezone.utc).strftime("%m%d%H%M")
ACCOUNT = boto3.client("sts").get_caller_identity()["Account"]

print(f"Account : {ACCOUNT}")
print(f"Region  : {REGION}")
print(f"Model   : {MODEL_ID}")
print(f"Run tag : {RUN}")

# ── OpenTelemetry setup ──
# InMemorySpanExporter captures spans in a list so we can analyse them
# right here in the notebook — no CloudWatch round-trip required.
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory import InMemorySpanExporter
from opentelemetry.sdk.resources import Resource

span_exporter = InMemorySpanExporter()
otel_resource = Resource.create({
    "service.name": "novapay-fraud-agent",
    "service.version": "3.2.0",
    "deployment.environment": "lab",
})
provider = TracerProvider(resource=otel_resource)
provider.add_span_processor(SimpleSpanProcessor(span_exporter))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer("novapay.observability.lab9")

print(f"\n✅ OpenTelemetry configured — spans captured in memory for analysis")

---
# LEVEL 100 · The Business Problem

## The "Dark Tuesday" Incident

**Tuesday, 03:14 UTC** — NovaPay's fraud-scoring agent processes a cross-border transfer:

| Field | Value |
|---|---|
| Transaction ID | TXN-DT-091 |
| Amount | ₦91,500,000 (~$91,500 USD) |
| Corridor | Nigeria → Ghana (NG-GH) |
| Sender | ACCT-4491 (flagged 3 times in 90 days) |
| Recipient | ACCT-GH-7832 (new account, opened 6 days ago) |

The agent calls `score_transaction_risk`. The risk score comes back **45**. The threshold
is 50. The agent approves the transfer.

**Wednesday, 09:22 UTC** — the correspondent bank in Accra flags the transaction as a
suspected layering pattern. The funds have already been withdrawn.

**What went wrong?** The risk scorer gave 45 because it calculated the base score from the
amount alone — it never received the sender's flag history. The sender had been flagged
**3 times** in the past 90 days. With that context, the score should have been **99**, and
the decision should have been **block**.

**Why it took 14 hours to figure this out:**

1. No trace — nobody could see which tools the agent called or in what order
2. Tool inputs and outputs were not logged — the team could not see that `sender_flags` was 0
3. No span attributes — even after finding the risk-score log line, nobody could tell *why* it was 45
4. No session correlation — 47 other decisions that hour; finding TXN-DT-091 meant full-text searching raw logs

**What this lab builds:** an observability layer where finding the root cause takes
15 minutes: look up the session → see the span tree → see `sender_flags=0` →
see that `check_sender_history` was not called before scoring → root cause found.

## Observability Concepts — Traces, Spans, and Metrics

| Concept | Traditional App | AI Agent | NovaPay Example |
|---|---|---|---|
| **Trace** | Request flowing through microservices | Session flowing through reasoning → tool calls → response | Full lifecycle of TXN-DT-091 from prompt to "approve" |
| **Span** | One service's processing | One tool call, one model invocation | The `score_transaction_risk` call with `sender_flags=0` |
| **Metric** | Request count, p99 latency | Tool call count, decision latency, approval rate | "35% of NG-GH transactions scored below 50 this week" |

### Why AI agent observability is harder than traditional observability

In a microservice, the call graph is **deterministic** — service A always calls B then C.
You know the expected sequence.

An AI agent's call graph is **non-deterministic**. The model decides which tools to call
and in what order. The same input can produce different tool sequences on different runs.
You cannot predict the span tree — you must **capture and inspect it after the fact**.

This is the fundamental reason you need instrumentation on every tool: the agent's
reasoning is opaque, but its tool calls are concrete and observable.

### OpenTelemetry (OTEL) — the standard

OpenTelemetry is the open standard for telemetry data. AgentCore uses it natively:

- **ADOT** (AWS Distro for OpenTelemetry) — the AWS-optimised OTEL distribution
- **CloudWatch** — stores spans, logs, and metrics
- **GenAI Observability Dashboard** — visualises agent traces in the CloudWatch console
- **In this lab** — we use the OTEL SDK locally (InMemorySpanExporter), then wire the same
  instrumentation through the AgentCore Gateway to CloudWatch

## Resource Lifecycle

| # | Resource | Created in | Ready signal | Teardown |
|---|---|---|---|---|
| 1 | IAM Role | 300.1 | immediate (10s propagation) | 400.5 |
| 2 | Lambda Function | 300.1 | `function_active_v2` waiter | 400.5 |
| 3 | Cognito User Pool | 300.2 | immediate | 400.5 |
| 4 | AgentCore Gateway | 300.3 | poll for READY/ACTIVE | 400.5 |
| 5 | Gateway Target | 300.4 | poll for ACTIVE | 400.5 |
| 6 | CW Log Group + Delivery | 300.3 | immediate | 400.5 |

> **Lifecycle warning**: Gateway creation is asynchronous. `create_gateway` returns status
> CREATING. Attempting to add targets before the gateway is READY will fail. This lab
> includes a `wait_for_gateway_ready()` poller — same pattern as Lab 8.

---
# LEVEL 200 · Instrumented NovaPay Tools

## 200.1 · Uninstrumented vs Instrumented

We build the same three NovaPay fraud tools twice:

1. **Version 1 (uninstrumented)** — returns a result and nothing else
2. **Version 2 (instrumented)** — wraps every operation in an OpenTelemetry span with
   structured attributes, events, and PII masking

The difference: when something goes wrong, v1 gives you a JSON blob and a prayer.
v2 gives you a complete decision tree with timing, inputs, outputs, and the exact
code path that was taken.

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel

# ── NovaPay Tools v1: No Instrumentation ──

@tool
def check_sender_history_v1(sender_id: str) -> str:
    """Look up a sender's transaction history and flag count."""
    senders = {
        "ACCT-4491": {"name": "Sender 4491", "total_txns": 142, "flags": 3,
                      "flag_reasons": ["velocity spike", "new recipient pattern", "round-amount"]},
        "ACCT-2200": {"name": "Sender 2200", "total_txns": 891, "flags": 0, "flag_reasons": []},
    }
    return json.dumps(senders.get(sender_id, {"name": "Unknown", "total_txns": 0, "flags": 0, "flag_reasons": []}))


@tool
def score_transaction_risk_v1(transaction_id: str, amount: float, corridor: str, sender_flags: int = 0) -> str:
    """Score fraud risk for a cross-border transaction.
    sender_flags: number of prior flags for the sender (default 0 if unknown)."""
    base_score = min(95, int(amount / 2000))
    flag_adjustment = sender_flags * 20
    final_score = min(99, base_score + flag_adjustment)
    decision = "approve" if final_score < 50 else ("review" if final_score < 80 else "block")
    return json.dumps({"transaction_id": transaction_id, "risk_score": final_score, "decision": decision})


@tool
def verify_corridor_compliance_v1(corridor: str, amount: float) -> str:
    """Check corridor-specific compliance rules."""
    corridors = {
        "NG-GH": {"max_single": 100000, "requires_sar": amount > 50000, "status": "open"},
        "NG-UK": {"max_single": 500000, "requires_sar": amount > 200000, "status": "open"},
    }
    info = corridors.get(corridor, {"max_single": 50000, "requires_sar": True, "status": "restricted"})
    info.update({"corridor": corridor, "amount": amount, "within_limit": amount <= info["max_single"]})
    return json.dumps(info)


model = BedrockModel(model_id=MODEL_ID, region_name=REGION)

DARK_TUESDAY_PROMPT = """Process this cross-border payment:
- Transaction ID: TXN-DT-091
- Amount: 91500 (NGN equivalent in USD)
- Corridor: NG-GH
- Sender: ACCT-4491
- Recipient: ACCT-GH-7832 (new account, 6 days old)
Decide: approve, review, or block."""

agent_v1 = Agent(
    model=model,
    tools=[check_sender_history_v1, score_transaction_risk_v1, verify_corridor_compliance_v1],
    system_prompt="You are NovaPay's fraud detection agent. For each transaction: "
    "1) Check the sender's history to get their flag count, "
    "2) Score the transaction risk (pass the sender flag count), "
    "3) Verify corridor compliance. Then decide: approve, review, or block.",
)

print("Running Dark Tuesday through the UNINSTRUMENTED agent...\n")
response_v1 = agent_v1(DARK_TUESDAY_PROMPT)
print(f"\n{'='*60}")
print("Agent decision:", str(response_v1)[:600])
print(f"\n⚠️  That is all we know. No trace. No span attributes.")
print(f"   If this decision is wrong, debugging starts from zero.")

## 200.2 · Adding OpenTelemetry Instrumentation

Now we build the same tools with custom spans. Every operation gets:

- A **span name** (e.g. `novapay.fraud_scoring`) for filtering
- **Attributes** for every input, intermediate value, and output
- **Events** for significant occurrences (flags found, warnings)
- **PII masking** — account IDs are truncated, no full names in spans
- **Timing** — automatically captured by the span start/end lifecycle

The instrumentation costs microseconds per span at runtime but saves hours during debugging.

In [ ]:
span_exporter.clear()  # start fresh

# ── NovaPay Tools v2: Fully Instrumented ──

@tool
def check_sender_history_v2(sender_id: str) -> str:
    """Look up a sender's transaction history and flag count. [Instrumented]"""
    with tracer.start_as_current_span("novapay.check_sender_history") as span:
        span.set_attribute("novapay.sender_id_masked", sender_id[:6] + "***")
        senders = {
            "ACCT-4491": {"name": "Sender 4491", "total_txns": 142, "flags": 3,
                          "flag_reasons": ["velocity spike", "new recipient pattern", "round-amount"]},
            "ACCT-2200": {"name": "Sender 2200", "total_txns": 891, "flags": 0, "flag_reasons": []},
        }
        record = senders.get(sender_id, {"name": "Unknown", "total_txns": 0, "flags": 0, "flag_reasons": []})
        span.set_attribute("novapay.flags_found", record["flags"])
        span.set_attribute("novapay.total_txns", record["total_txns"])
        span.set_attribute("novapay.has_prior_flags", record["flags"] > 0)
        if record["flags"] > 0:
            span.add_event("sender_flagged", {"flag_count": record["flags"],
                                               "reasons": str(record["flag_reasons"])})
        return json.dumps(record)


@tool
def score_transaction_risk_v2(transaction_id: str, amount: float, corridor: str, sender_flags: int = 0) -> str:
    """Score fraud risk for a cross-border transaction. [Instrumented]
    sender_flags: number of prior flags for the sender (default 0 if unknown)."""
    with tracer.start_as_current_span("novapay.fraud_scoring") as span:
        span.set_attribute("novapay.txn_id", transaction_id)
        span.set_attribute("novapay.amount", amount)
        span.set_attribute("novapay.corridor", corridor)
        span.set_attribute("novapay.sender_flags_input", sender_flags)

        base_score = min(95, int(amount / 2000))
        span.set_attribute("novapay.base_score", base_score)

        flag_adjustment = sender_flags * 20
        span.set_attribute("novapay.flag_adjustment", flag_adjustment)
        span.set_attribute("novapay.used_default_flags", sender_flags == 0)

        final_score = min(99, base_score + flag_adjustment)
        span.set_attribute("novapay.final_score", final_score)

        decision = "approve" if final_score < 50 else ("review" if final_score < 80 else "block")
        span.set_attribute("novapay.decision", decision)

        if sender_flags == 0 and amount > 50000:
            span.add_event("risk_warning", {
                "message": "High-value txn scored with default sender_flags=0",
                "recommendation": "Call check_sender_history before scoring",
            })
        return json.dumps({"transaction_id": transaction_id, "risk_score": final_score,
                           "decision": decision, "scoring_model": "novapay-fraud-v3.2"})


@tool
def verify_corridor_compliance_v2(corridor: str, amount: float) -> str:
    """Check corridor-specific compliance rules. [Instrumented]"""
    with tracer.start_as_current_span("novapay.corridor_compliance") as span:
        span.set_attribute("novapay.corridor", corridor)
        span.set_attribute("novapay.amount", amount)
        corridors = {
            "NG-GH": {"max_single": 100000, "requires_sar": amount > 50000, "status": "open"},
            "NG-UK": {"max_single": 500000, "requires_sar": amount > 200000, "status": "open"},
        }
        info = corridors.get(corridor, {"max_single": 50000, "requires_sar": True, "status": "restricted"})
        info.update({"corridor": corridor, "amount": amount, "within_limit": amount <= info["max_single"]})
        span.set_attribute("novapay.within_limit", info["within_limit"])
        span.set_attribute("novapay.requires_sar", info.get("requires_sar", False))
        span.set_attribute("novapay.corridor_status", info["status"])
        return json.dumps(info)

print("✅ Instrumented tools (v2) ready")

In [ ]:
agent_v2 = Agent(
    model=model,
    tools=[check_sender_history_v2, score_transaction_risk_v2, verify_corridor_compliance_v2],
    system_prompt="You are NovaPay's fraud detection agent. For each transaction: "
    "1) Check the sender's history to get their flag count, "
    "2) Score the transaction risk (pass the sender flag count), "
    "3) Verify corridor compliance. Then decide: approve, review, or block.",
)

print("Running Dark Tuesday through the INSTRUMENTED agent...\n")
response_v2 = agent_v2(DARK_TUESDAY_PROMPT)
print(f"\n{'='*60}")
print("Agent decision:", str(response_v2)[:600])

In [ ]:
# ── The 15-minute investigation ──
# What took NovaPay 14 hours without tracing now takes one cell.

spans = span_exporter.get_finished_spans()
print(f"Captured {len(spans)} span(s) from the instrumented run\n")

for i, s in enumerate(spans):
    dur = (s.end_time - s.start_time) / 1_000_000
    print(f"─"*60)
    print(f"SPAN {i+1}: {s.name}  ({dur:.1f} ms)")
    if s.attributes:
        for k, v in s.attributes.items():
            tag = ""
            if k == "novapay.used_default_flags" and v is True:
                tag = "  ← ⚠️ ROOT CAUSE"
            if k == "novapay.sender_flags_input" and v == 0:
                tag = "  ← ⚠️ default value"
            if k == "novapay.has_prior_flags" and v is True:
                tag = "  ← ⚡ sender HAS flags"
            print(f"  {k} = {v}{tag}")
    if s.events:
        for e in s.events:
            print(f"  EVENT [{e.name}]: {dict(e.attributes)}")

# ── Summary ──
print(f"\n{'='*60}\nINVESTIGATION SUMMARY\n{'='*60}")
scoring = [s for s in spans if s.name == "novapay.fraud_scoring"]
history = [s for s in spans if s.name == "novapay.check_sender_history"]

if scoring:
    ss = scoring[0]
    flags_in = ss.attributes.get("novapay.sender_flags_input", "?")
    used_def = ss.attributes.get("novapay.used_default_flags", "?")
    score    = ss.attributes.get("novapay.final_score", "?")
    decision = ss.attributes.get("novapay.decision", "?")
    print(f"\nScorer received sender_flags = {flags_in}")
    print(f"Used default (no real data)? = {used_def}")
    print(f"Final risk score             = {score}")
    print(f"Decision                     = {decision}")

if history:
    hs = history[0]
    actual = hs.attributes.get("novapay.flags_found", "?")
    print(f"\nSender history: actual flags  = {actual}")
    if scoring and actual != scoring[0].attributes.get("novapay.sender_flags_input", 0):
        correct = min(99, min(95, int(91500/2000)) + int(actual) * 20)
        print(f"⚠️  MISMATCH: history shows {actual} flags but scorer received "
              f"{scoring[0].attributes.get('novapay.sender_flags_input', 0)}")
        cdec = "approve" if correct < 50 else ("review" if correct < 80 else "block")
        print(f"   Correct score with {actual} flags: {correct} → {cdec}")
    else:
        print(f"✅ Flag count correctly propagated to scorer.")
elif not history:
    print(f"\n⚠️  check_sender_history was NEVER called!")

print(f"\n⏱️  Time to root cause: ~2 minutes (span attributes)")
print(f"   Without instrumentation: ~14 hours (raw log search)")

> ### Level 200 checkpoint
> With instrumented tools, every decision is **traceable**: inputs, outputs, intermediate
> values, timing, and warnings are captured as structured span attributes.
>
> The Dark Tuesday root cause is visible in one attribute: `novapay.used_default_flags = True`.
>
> **Key principle:** You cannot debug what you cannot see. Instrumentation makes the
> agent's decision process visible *after the fact*, without changing the agent's behaviour.

---
# LEVEL 300 · End-to-End Tracing Through the Gateway

## Why local tracing is not enough

Level 200 captured spans locally using InMemorySpanExporter. That works for development.
In production, NovaPay's agents run behind an AgentCore Gateway (Lab 8). The gateway adds
its own spans — routing decisions, authorisation checks, target invocations — and all of
that telemetry needs to flow to CloudWatch alongside the agent's tool spans.

This level creates a real gateway with **tracing and log delivery enabled**, runs the Dark
Tuesday scenario through it, and shows how to reconstruct the full distributed trace:
agent reasoning → gateway routing → Lambda tool execution.

## 300.1 · Create the fraud backend (IAM Role + Lambda)

In [ ]:
iam = boto3.client("iam")
lam = boto3.client("lambda", region_name=REGION)

ROLE_NAME = f"NovaPay-Lab9-Lambda-{RUN}"
LAMBDA_NAME = f"novapay-fraud-tools-obs-{RUN}"

# ── IAM Role ──
TRUST = json.dumps({
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "lambda.amazonaws.com"},
        "Action": "sts:AssumeRole",
    }]
})

try:
    role = iam.create_role(RoleName=ROLE_NAME, AssumeRolePolicyDocument=TRUST,
                           Description="Lab 9 Lambda execution role")
    ROLE_ARN = role["Role"]["Arn"]
    iam.attach_role_policy(RoleName=ROLE_NAME,
                           PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole")
    print(f"✅ IAM role created: {ROLE_ARN}")
except iam.exceptions.EntityAlreadyExistsException:
    ROLE_ARN = iam.get_role(RoleName=ROLE_NAME)["Role"]["Arn"]
    print(f"ℹ️  IAM role exists: {ROLE_ARN}")

time.sleep(10)  # IAM propagation

# ── Lambda source ──
LAMBDA_CODE = '''import json

def handler(event, context):
    # Extract tool name from AgentCore context
    raw = ""
    cc = getattr(context, "client_context", None)
    if cc is not None and getattr(cc, "custom", None):
        raw = cc.custom.get("bedrockAgentCoreToolName", "") or ""
    delimiter = "___"
    tool_name = raw.split(delimiter)[-1] if delimiter in raw else raw

    body = event if isinstance(event, dict) else json.loads(event) if isinstance(event, str) else {}

    if tool_name == "check_sender_history":
        sid = body.get("sender_id", "UNKNOWN")
        db = {"ACCT-4491": {"flags": 3, "total_txns": 142},
              "ACCT-2200": {"flags": 0, "total_txns": 891}}
        return db.get(sid, {"flags": 0, "total_txns": 0})

    if tool_name == "score_transaction_risk":
        amount = body.get("amount", 0)
        sender_flags = body.get("sender_flags", 0)
        base = min(95, int(amount / 2000))
        adj = sender_flags * 20
        final = min(99, base + adj)
        dec = "approve" if final < 50 else ("review" if final < 80 else "block")
        return {"transaction_id": body.get("transaction_id", "?"),
                "risk_score": final, "decision": dec}

    if tool_name == "verify_corridor_compliance":
        corridor = body.get("corridor", "?")
        amount = body.get("amount", 0)
        rules = {"NG-GH": {"max": 100000, "sar": amount > 50000},
                 "NG-UK": {"max": 500000, "sar": amount > 200000}}
        r = rules.get(corridor, {"max": 50000, "sar": True})
        return {"corridor": corridor, "within_limit": amount <= r["max"],
                "requires_sar": r["sar"]}

    return {"error": "Unknown tool: " + tool_name}
'''

# ── Package and create ──
buf = io.BytesIO()
with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("lambda_function.py", LAMBDA_CODE)
code_bytes = buf.getvalue()

try:
    fn = lam.create_function(
        FunctionName=LAMBDA_NAME, Runtime="python3.12", Role=ROLE_ARN,
        Handler="lambda_function.handler", Code={"ZipFile": code_bytes},
        Timeout=30, MemorySize=256,
        Description="NovaPay fraud tools for Lab 9 (observability)",
    )
    LAMBDA_ARN = fn["FunctionArn"]
    print(f"✅ Lambda created: {LAMBDA_NAME}")
except lam.exceptions.ResourceConflictException:
    LAMBDA_ARN = lam.get_function(FunctionName=LAMBDA_NAME)["Configuration"]["FunctionArn"]
    print(f"ℹ️  Lambda exists: {LAMBDA_NAME}")

# Wait for Lambda to become active
try:
    lam.get_waiter("function_active_v2").wait(FunctionName=LAMBDA_NAME,
                                               WaiterConfig={"Delay": 5, "MaxAttempts": 30})
except Exception:
    time.sleep(15)
print(f"✅ Lambda active: {LAMBDA_ARN}")

## 300.2 · Identity provider (Cognito)

As in Lab 8, the gateway needs a JWT authoriser. We create a minimal Cognito user pool
with one scope (`fraud-ops/all`) and one M2M client. The focus here is observability,
not identity — see Lab 8 for the full least-privilege model.

In [ ]:
cognito = boto3.client("cognito-idp", region_name=REGION)

POOL_NAME = f"novapay-obs-pool-{RUN}"
pool = cognito.create_user_pool(PoolName=POOL_NAME, AdminCreateUserConfig={"AllowAdminCreateUserOnly": True})
POOL_ID = pool["UserPool"]["Id"]
print(f"✅ Cognito pool: {POOL_ID}")

cognito.create_resource_server(
    UserPoolId=POOL_ID, Identifier="fraud-ops", Name="Fraud Ops",
    Scopes=[{"ScopeName": "all", "ScopeDescription": "Full fraud-ops access"}],
)

COGNITO_DOMAIN = f"novapay-obs-{RUN}".lower()
try:
    cognito.create_user_pool_domain(Domain=COGNITO_DOMAIN, UserPoolId=POOL_ID)
except Exception:
    COGNITO_DOMAIN = f"novapay-obs-{RUN}-alt".lower()
    cognito.create_user_pool_domain(Domain=COGNITO_DOMAIN, UserPoolId=POOL_ID)

client_resp = cognito.create_user_pool_client(
    UserPoolId=POOL_ID, ClientName="fraud-ops-agent",
    GenerateSecret=True, AllowedOAuthFlows=["client_credentials"],
    AllowedOAuthScopes=["fraud-ops/all"],
    AllowedOAuthFlowsUserPoolClient=True,
)
CLIENT_ID = client_resp["UserPoolClient"]["ClientId"]
CLIENT_SECRET = client_resp["UserPoolClient"]["ClientSecret"]
TOKEN_URL = f"https://{COGNITO_DOMAIN}.auth.{REGION}.amazoncognito.com/oauth2/token"
ISSUER = f"https://cognito-idp.{REGION}.amazonaws.com/{POOL_ID}"

print(f"✅ Client ID : {CLIENT_ID}")
print(f"✅ Issuer    : {ISSUER}")
print(f"✅ Token URL : {TOKEN_URL}")

## 300.3 · Create the observable gateway

This is where Lab 9 diverges from Lab 8. We create the gateway with **tracing enabled**
and configure **log delivery to CloudWatch**. This means every tool call through the
gateway produces spans and structured logs that are queryable in CloudWatch Transaction
Search and visible in the GenAI Observability dashboard.

In [ ]:
agentcore = boto3.client("bedrock-agentcore-control", region_name=REGION)
logs_client = boto3.client("logs", region_name=REGION)

GATEWAY_NAME = f"novapay-obs-gw-{RUN}"

# ── Create Gateway ──
gw = agentcore.create_gateway(
    name=GATEWAY_NAME,
    protocolType="MCP",
    description="NovaPay fraud gateway with observability (Lab 9)",
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration={
        "customJWTAuthorizerConfiguration": {
            "allowedAudiences": [CLIENT_ID],
            "allowedIssuers": [ISSUER],
        }
    },
    searchType="SEMANTIC",
)
GATEWAY_ID = gw["gatewayId"]
GATEWAY_ARN = gw.get("gatewayArn", f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT}:gateway/{GATEWAY_ID}")
print(f"✅ Gateway created: {GATEWAY_ID} (status: {gw.get('status', 'CREATING')})")

# ── Wait for gateway to become ready ──
def wait_for_gateway_ready(gw_id, timeout=300, interval=10):
    started = time.time()
    while time.time() - started < timeout:
        status = agentcore.get_gateway(gatewayIdentifier=gw_id).get("status", "UNKNOWN")
        if status in ("READY", "ACTIVE", "AVAILABLE"):
            return status
        if status in ("FAILED", "DELETING"):
            raise RuntimeError(f"Gateway entered terminal status: {status}")
        print(f"   gateway status: {status} … waiting")
        time.sleep(interval)
    raise TimeoutError(f"Gateway did not become ready within {timeout}s")

final_status = wait_for_gateway_ready(GATEWAY_ID)
GATEWAY_URL = agentcore.get_gateway(gatewayIdentifier=GATEWAY_ID).get("gatewayUrl", "")
print(f"✅ Gateway ready ({final_status})")
print(f"   URL: {GATEWAY_URL}")

In [ ]:
# ── Configure CloudWatch log delivery for the gateway ──
# This is the observability wiring that makes gateway calls visible.

LOG_GROUP = f"/aws/vendedlogs/bedrock-agentcore/gateway/{GATEWAY_ID}"

try:
    logs_client.create_log_group(logGroupName=LOG_GROUP)
    print(f"✅ Log group created: {LOG_GROUP}")
except logs_client.exceptions.ResourceAlreadyExistsException:
    print(f"ℹ️  Log group exists: {LOG_GROUP}")

LOG_GROUP_ARN = f"arn:aws:logs:{REGION}:{ACCOUNT}:log-group:{LOG_GROUP}"

# Step 1: Create delivery source (the gateway)
try:
    logs_client.put_delivery_source(
        name=f"{GATEWAY_ID}-logs-source",
        logType="APPLICATION_LOGS",
        resourceArn=GATEWAY_ARN,
    )
    print(f"✅ Delivery source created")
except Exception as e:
    print(f"ℹ️  Delivery source: {e}")

# Step 2: Create delivery destination (CloudWatch log group)
try:
    logs_client.put_delivery_destination(
        name=f"{GATEWAY_ID}-logs-dest",
        deliveryDestinationType="CWL",
        deliveryDestinationConfiguration={"destinationResourceArn": LOG_GROUP_ARN},
    )
    print(f"✅ Delivery destination created")
except Exception as e:
    print(f"ℹ️  Delivery destination: {e}")

# Step 3: Connect source to destination
try:
    logs_client.create_delivery(
        deliverySourceName=f"{GATEWAY_ID}-logs-source",
        deliveryDestinationArn=f"arn:aws:logs:{REGION}:{ACCOUNT}:delivery-destination:{GATEWAY_ID}-logs-dest",
    )
    print(f"✅ Log delivery pipeline connected: gateway → CloudWatch")
except Exception as e:
    print(f"ℹ️  Delivery pipeline: {e}")

# Step 4: Create trace delivery source and destination
try:
    logs_client.put_delivery_source(
        name=f"{GATEWAY_ID}-traces-source",
        logType="TRACES",
        resourceArn=GATEWAY_ARN,
    )
    logs_client.put_delivery_destination(
        name=f"{GATEWAY_ID}-traces-dest",
        deliveryDestinationType="XRAY",
    )
    logs_client.create_delivery(
        deliverySourceName=f"{GATEWAY_ID}-traces-source",
        deliveryDestinationArn=f"arn:aws:logs:{REGION}:{ACCOUNT}:delivery-destination:{GATEWAY_ID}-traces-dest",
    )
    print(f"✅ Trace delivery pipeline connected: gateway → X-Ray → CloudWatch")
except Exception as e:
    print(f"ℹ️  Trace delivery: {e}")

print(f"\n✅ Gateway observability fully configured")
print(f"   Logs : {LOG_GROUP}")
print(f"   Spans: CloudWatch Transaction Search / GenAI Observability Dashboard")

In [ ]:
# ── Add the Lambda as a gateway target ──

TARGET_NAME = f"novapay-fraud-backend-{RUN}"

# Grant the gateway permission to invoke the Lambda
try:
    lam.add_permission(
        FunctionName=LAMBDA_NAME,
        StatementId=f"gw-invoke-{RUN}",
        Action="lambda:InvokeFunction",
        Principal="bedrock-agentcore.amazonaws.com",
        SourceArn=GATEWAY_ARN,
    )
except lam.exceptions.ResourceConflictException:
    pass

TOOLS_SCHEMA = [
    {
        "name": "check_sender_history",
        "description": "Look up a sender's transaction history and prior fraud flags",
        "inputSchema": {"json": {"type": "object",
            "properties": {"sender_id": {"type": "string", "description": "Sender account ID"}},
            "required": ["sender_id"]}},
    },
    {
        "name": "score_transaction_risk",
        "description": "Score fraud risk for a cross-border transaction",
        "inputSchema": {"json": {"type": "object",
            "properties": {
                "transaction_id": {"type": "string"},
                "amount": {"type": "number", "description": "Transaction amount"},
                "corridor": {"type": "string", "description": "Payment corridor e.g. NG-GH"},
                "sender_flags": {"type": "integer", "description": "Number of prior flags for the sender (default 0)"},
            },
            "required": ["transaction_id", "amount", "corridor"]}},
    },
    {
        "name": "verify_corridor_compliance",
        "description": "Check corridor-specific regulatory compliance rules",
        "inputSchema": {"json": {"type": "object",
            "properties": {
                "corridor": {"type": "string"},
                "amount": {"type": "number"},
            },
            "required": ["corridor", "amount"]}},
    },
]

target = agentcore.create_gateway_target(
    gatewayIdentifier=GATEWAY_ID,
    name=TARGET_NAME,
    description="NovaPay fraud detection tools (3 tools, instrumented backend)",
    targetConfiguration={
        "lambdaTargetConfiguration": {
            "lambdaArn": LAMBDA_ARN,
            "toolSchema": {"tools": TOOLS_SCHEMA},
        }
    },
)
TARGET_ID = target.get("targetId", TARGET_NAME)
print(f"✅ Gateway target created: {TARGET_ID}")

# Wait for target to become active
started = time.time()
while time.time() - started < 120:
    t = agentcore.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
    st = t.get("status", "UNKNOWN")
    if st in ("ACTIVE", "AVAILABLE", "READY"):
        print(f"✅ Target active")
        break
    if st in ("FAILED",):
        raise RuntimeError(f"Target failed: {t}")
    time.sleep(8)
else:
    print(f"⚠️  Target status: {st} (proceeding anyway)")

## 300.4 · Dark Tuesday replay through the observable gateway

Now we replay the Dark Tuesday scenario through the gateway. Every call produces:

1. **Agent-side spans** (our InMemorySpanExporter) — tool-level decision data
2. **Gateway spans** (CloudWatch) — routing, auth check, target invocation timing
3. **Lambda logs** (CloudWatch) — tool execution on the backend

Together, these three layers give you **distributed tracing** across the entire decision
pipeline. In the GenAI Observability dashboard, you can click through from the session
to the agent trace to the individual tool span.

In [ ]:
import requests
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client
from strands.tools.mcp.mcp_client import MCPClient
import base64

# ── Get an access token ──
def get_token(client_id, client_secret, scopes):
    creds = base64.b64encode(f"{client_id}:{client_secret}".encode()).decode()
    r = requests.post(TOKEN_URL, headers={
        "Content-Type": "application/x-www-form-urlencoded",
        "Authorization": f"Basic {creds}",
    }, data={"grant_type": "client_credentials", "scope": " ".join(scopes)})
    r.raise_for_status()
    return r.json()["access_token"]

token = get_token(CLIENT_ID, CLIENT_SECRET, ["fraud-ops/all"])
print(f"✅ Token obtained (first 20 chars): {token[:20]}...")

# ── Connect to gateway via MCP ──
def gateway_mcp(access_token):
    return MCPClient(lambda: streamablehttp_client(
        GATEWAY_URL, headers={"Authorization": f"Bearer {access_token}"}
    ))

span_exporter.clear()  # fresh span collection

SESSION_ID = f"dark-tuesday-replay-{RUN}"

with gateway_mcp(token) as mcp:
    tools = mcp.list_tools_sync()
    print(f"\n✅ Gateway advertises {len(tools)} tool(s):")
    for t in tools:
        name = getattr(t, "tool_name", getattr(t, "name", str(t)))
        print(f"   • {name}")

    # ── Replay the Dark Tuesday flow ──
    # Step 1: Check sender history (this SHOULD happen first)
    print(f"\n▶️  Step 1: check_sender_history for ACCT-4491")
    with tracer.start_as_current_span("dark_tuesday.check_sender", attributes={"session.id": SESSION_ID}):
        hist = mcp.call_tool_sync(
            tool_use_id=str(uuid.uuid4()),
            name=next(getattr(t, "tool_name", getattr(t, "name", "")) for t in tools
                      if "check_sender" in getattr(t, "tool_name", getattr(t, "name", ""))),
            arguments={"sender_id": "ACCT-4491"},
        )
    hist_data = json.loads(str(hist)) if isinstance(hist, str) else hist
    print(f"   Result: {json.dumps(hist_data, default=str)[:300]}")

    # Step 2: Score risk — the critical call
    print(f"\n▶️  Step 2: score_transaction_risk for TXN-DT-091")
    # Simulate the Dark Tuesday bug: agent does NOT pass sender_flags
    with tracer.start_as_current_span("dark_tuesday.score_risk", attributes={
        "session.id": SESSION_ID,
        "novapay.bug_simulated": "sender_flags not passed from history"
    }):
        score_result = mcp.call_tool_sync(
            tool_use_id=str(uuid.uuid4()),
            name=next(getattr(t, "tool_name", getattr(t, "name", "")) for t in tools
                      if "score_transaction" in getattr(t, "tool_name", getattr(t, "name", ""))),
            arguments={"transaction_id": "TXN-DT-091", "amount": 91500, "corridor": "NG-GH"},
            # Note: sender_flags is NOT passed — this is the bug
        )
    print(f"   Result: {json.dumps(score_result, default=str)[:300]}")

    # Step 3: Corridor compliance
    print(f"\n▶️  Step 3: verify_corridor_compliance for NG-GH")
    with tracer.start_as_current_span("dark_tuesday.corridor_check", attributes={"session.id": SESSION_ID}):
        corridor = mcp.call_tool_sync(
            tool_use_id=str(uuid.uuid4()),
            name=next(getattr(t, "tool_name", getattr(t, "name", "")) for t in tools
                      if "corridor" in getattr(t, "tool_name", getattr(t, "name", ""))),
            arguments={"corridor": "NG-GH", "amount": 91500},
        )
    print(f"   Result: {json.dumps(corridor, default=str)[:300]}")

    print(f"\n{'='*60}")
    print(f"Dark Tuesday replay complete. Session: {SESSION_ID}")
    print(f"Gateway spans are flowing to CloudWatch: {LOG_GROUP}")

In [ ]:
# ── Reconstruct the incident from our spans ──
# In production, you would query CloudWatch. Here, we use our InMemorySpanExporter.
# The pattern is the same: look up by session, read the span tree.

replay_spans = span_exporter.get_finished_spans()
print(f"INCIDENT RECONSTRUCTION: Session {SESSION_ID}")
print(f"{'='*60}\n")

# Build a timeline
timeline = []
for s in replay_spans:
    start_ns = s.start_time
    dur_ms = (s.end_time - s.start_time) / 1_000_000
    timeline.append((start_ns, s.name, dur_ms, dict(s.attributes or {})))

timeline.sort(key=lambda x: x[0])

print(f"{'Step':<6} {'Span Name':<35} {'Duration':>10}  Key Attributes")
print(f"{'----':<6} {'-'*35:<35} {'-'*10:>10}  {'-'*40}")

for i, (_, name, dur, attrs) in enumerate(timeline, 1):
    key_attrs = {k: v for k, v in attrs.items() if k.startswith("novapay.") or k == "session.id"}
    print(f"{i:<6} {name:<35} {dur:>8.1f}ms  {key_attrs}")

# ── The smoking gun ──
print(f"\n{'='*60}")
print("ROOT CAUSE ANALYSIS")
print(f"{'='*60}")
print(f"""
1. check_sender_history WAS called and returned flags=3
2. score_transaction_risk was called WITHOUT passing sender_flags
   (the default value 0 was used)
3. Risk score: base={min(95, int(91500/2000))} + flag_adj=0 = {min(95, int(91500/2000))}
   Decision: approve (below 50 threshold)
4. CORRECT calculation: base={min(95, int(91500/2000))} + flag_adj={3*20} = {min(99, min(95, int(91500/2000))+60)}
   Correct decision: block

FIX: Make sender_flags a required parameter in score_transaction_risk,
     or have the tool internally call check_sender_history.

DETECTION TIME: ~5 minutes (reading span attributes)
PREVIOUS METHOD: ~14 hours (searching raw CloudWatch logs)
""")

> ### Level 300 checkpoint
> You have wired observability through the AgentCore Gateway:
>
> - **Log delivery**: gateway logs flow to a CloudWatch log group
> - **Trace delivery**: gateway spans flow to CloudWatch Transaction Search
> - **Session correlation**: every tool call carries a session ID for cross-span lookup
>
> The distributed trace now spans: agent reasoning → gateway routing → Lambda execution.
> The Dark Tuesday root cause was found by reading 3 span attributes — no log diving required.

---
# LEVEL 400 · Production Operations

## 400.1 · CloudWatch Alarms — Catch It Before Compliance Does

The Dark Tuesday incident was discovered by the correspondent bank 30 hours later.
With proper alerting, NovaPay's ops team should have been paged within minutes.

AgentCore emits metrics to the `bedrock-agentcore` CloudWatch namespace. Key metrics:

| Metric | Namespace | What it catches |
|---|---|---|
| `Invocations` | bedrock-agentcore | Unexpected volume spike or drop |
| `Errors` | bedrock-agentcore | Tool failures, gateway errors |
| `Duration` | bedrock-agentcore | Latency spikes (model overload) |
| Custom: `FraudApprovalRate` | novapay-agents | Approval rate drift (should be ~60%, alert at >80%) |

In [ ]:
# ── Set up a CloudWatch alarm for error rate ──
cw = boto3.client("cloudwatch", region_name=REGION)

# Alarm: if gateway errors exceed 5 in 5 minutes, alert
ALARM_NAME = f"novapay-gw-errors-{RUN}"
try:
    cw.put_metric_alarm(
        AlarmName=ALARM_NAME,
        AlarmDescription="NovaPay gateway error rate exceeds threshold",
        Namespace="bedrock-agentcore",
        MetricName="Errors",
        Dimensions=[{"Name": "GatewayId", "Value": GATEWAY_ID}],
        Statistic="Sum",
        Period=300,
        EvaluationPeriods=1,
        Threshold=5,
        ComparisonOperator="GreaterThanThreshold",
        TreatMissingData="notBreaching",
        # In production, add: AlarmActions=["arn:aws:sns:..."]
    )
    print(f"✅ Alarm created: {ALARM_NAME}")
    print(f"   Triggers when: >5 gateway errors in 5 minutes")
    print(f"   In production: wire to SNS → PagerDuty/Slack")
except Exception as e:
    print(f"ℹ️  Alarm: {e}")

# ── Custom metric: publish a simulated approval rate ──
try:
    cw.put_metric_data(
        Namespace="novapay-agents",
        MetricData=[{
            "MetricName": "FraudApprovalRate",
            "Value": 0.73,
            "Unit": "None",
            "Dimensions": [{"Name": "Corridor", "Value": "NG-GH"},
                           {"Name": "AgentVersion", "Value": "v3.2"}],
        }],
    )
    print(f"✅ Custom metric published: FraudApprovalRate = 73%")
except Exception as e:
    print(f"ℹ️  Metric: {e}")

## 400.2 · Sampling — You Cannot Trace Everything

In production, tracing every session is expensive. CloudWatch charges per GB of log
data ingested. NovaPay processes ~12,000 sessions/day. At ~2 KB per traced session,
that is ~24 MB/day — manageable. But at full verbosity with model I/O, it could be
200 KB per session = 2.4 GB/day.

### Sampling strategies

| Strategy | When | How |
|---|---|---|
| **1% default** | Baseline coverage | CloudWatch Transaction Search default |
| **Head-based** | Sample at session start | `DesiredSamplingPercentage: 10` in UpdateIndexingRule |
| **Tail-based** | Keep traces with errors | ADOT Tail Sampling processor (export only if error span found) |
| **Always-on for high value** | Transactions > $50K | Set `Sampled=1` in `X-Amzn-Trace-Id` header |

NovaPay's recommended config: 10% baseline + always-on for transactions over $50,000.

In [ ]:
# ── Configure sampling (reference code — requires X-Ray permissions) ──
xray = boto3.client("xray", region_name=REGION)

try:
    xray.update_indexing_rule(
        Name="Default",
        Rule={"Probabilistic": {"DesiredSamplingPercentage": 10}},
    )
    print(f"✅ Sampling set to 10% (up from default 1%)")
    print(f"   High-value transactions: set Sampled=1 in trace header for 100% capture")
except Exception as e:
    print(f"ℹ️  Sampling config: {e}")
    print(f"   This requires X-Ray permissions. In production, run once during setup.")

## 400.3 · Cost Analysis

| Component | Unit | NovaPay Usage | Monthly Cost |
|---|---|---|---|
| CloudWatch Logs (spans) | $0.50/GB ingested | ~720 MB (10% sampling, 12K sessions/day) | ~$0.36 |
| CloudWatch Logs (gateway logs) | $0.50/GB ingested | ~1.5 GB | ~$0.75 |
| CloudWatch Metrics | $0.30/metric/month | 10 custom metrics | ~$3.00 |
| CloudWatch Alarms | $0.10/alarm/month | 5 alarms | ~$0.50 |
| X-Ray trace storage | $5.00/million traces stored | ~36K traces/month | ~$0.18 |
| **Total** | | | **~$4.79/month** |

**ROI calculation:** The Dark Tuesday incident cost NovaPay $91,500 in fraud losses plus
14 hours of engineering time (~$2,800 at loaded cost). Observability costs $5/month and
reduces detection time from 30 hours to 5 minutes. At even one prevented incident per
quarter, the ROI is >4,000x.

## 400.4 · Incident Response Workflow

When a CloudWatch alarm fires, NovaPay's on-call engineer follows this workflow:

| Step | Action | Tool |
|---|---|---|
| 1 | **Identify the session** | CloudWatch alarm → metric dimension → session ID |
| 2 | **Pull the trace** | GenAI Observability Dashboard → Traces View → filter by session |
| 3 | **Read the span tree** | Click through: agent → gateway → tool spans |
| 4 | **Find the anomaly** | Look for: unexpected attributes, missing spans, error status |
| 5 | **Root cause** | Span attributes tell you exactly what the tool received and returned |
| 6 | **Fix and verify** | Deploy fix, re-run with same inputs, compare span trees |

In the Dark Tuesday case:
- Step 1: Alarm fires on `FraudApprovalRate > 80%` for NG-GH corridor
- Step 2: Filter traces for NG-GH corridor, sort by amount descending
- Step 3: TXN-DT-091 trace shows: check_sender_history → score_transaction_risk → corridor
- Step 4: `novapay.sender_flags_input = 0` and `novapay.used_default_flags = True`
- Step 5: Sender flags were not propagated from history check to scorer
- Step 6: Make sender_flags required, redeploy, verify with same transaction

In [ ]:
# ── 400.4b · Session correlation with OpenTelemetry baggage ──
# In production, propagate session ID via OTEL baggage so all spans
# (across agents, gateways, and tools) share the same session context.

from opentelemetry import baggage, context

session_id = f"novapay-session-{uuid.uuid4().hex[:8]}"
ctx = baggage.set_baggage("session.id", session_id)

# All spans created within this context will inherit the session ID
token = context.attach(ctx)
try:
    with tracer.start_as_current_span("novapay.session_demo") as span:
        span.set_attribute("session.id", session_id)
        span.set_attribute("novapay.demo", "session correlation")
        print(f"✅ Session correlation demo")
        print(f"   session.id: {session_id}")
        print(f"   All spans in this context share this ID")
        print(f"   In CloudWatch: filter by session.id to see the full trace")
finally:
    context.detach(token)

print(f"\n   Production pattern:")
print(f"   1. Generate session ID at agent entry point")
print(f"   2. Set as OTEL baggage: baggage.set_baggage('session.id', sid)")
print(f"   3. Pass in gateway header: X-Amzn-Bedrock-AgentCore-Runtime-Session-Id")
print(f"   4. All downstream spans inherit the session context")

> ### Level 400 checkpoint
> You now have the full production observability stack:
>
> - **Alerting**: CloudWatch alarms on error rates and custom metrics
> - **Sampling**: 10% baseline + always-on for high-value transactions
> - **Cost**: ~$5/month for full observability on 12,000 sessions/day
> - **Incident workflow**: alarm → trace → span tree → root cause in minutes
> - **Session correlation**: OTEL baggage propagates session ID across all spans

## 400.5 · Teardown

Delete every resource created in this lab. Run this cell to avoid ongoing charges.

In [ ]:
errors = []

def safe(fn, label):
    try:
        fn()
        print(f"✅ {label}")
    except Exception as e:
        errors.append((label, str(e)))
        print(f"⚠️  {label}: {e}")

# 1. Gateway target
safe(lambda: agentcore.delete_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID),
     "Delete gateway target")

time.sleep(5)

# 2. Gateway
safe(lambda: agentcore.delete_gateway(gatewayIdentifier=GATEWAY_ID),
     "Delete gateway")

# 3. CloudWatch alarm
safe(lambda: cw.delete_alarms(AlarmNames=[ALARM_NAME]),
     "Delete CloudWatch alarm")

# 4. Log deliveries (best-effort — may already be cleaned up with gateway)
for suffix in ["logs", "traces"]:
    safe(lambda s=suffix: logs_client.delete_delivery_source(name=f"{GATEWAY_ID}-{s}-source"),
         f"Delete {suffix} delivery source")
    safe(lambda s=suffix: logs_client.delete_delivery_destination(name=f"{GATEWAY_ID}-{s}-dest"),
         f"Delete {suffix} delivery destination")

# 5. Log group
safe(lambda: logs_client.delete_log_group(logGroupName=LOG_GROUP),
     "Delete log group")

# 6. Lambda
safe(lambda: lam.delete_function(FunctionName=LAMBDA_NAME),
     "Delete Lambda")

# 7. Cognito
safe(lambda: cognito.delete_user_pool_domain(Domain=COGNITO_DOMAIN, UserPoolId=POOL_ID),
     "Delete Cognito domain")
safe(lambda: cognito.delete_user_pool(UserPoolId=POOL_ID),
     "Delete Cognito pool")

# 8. IAM
for p in ["arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole"]:
    safe(lambda p=p: iam.detach_role_policy(RoleName=ROLE_NAME, PolicyArn=p),
         f"Detach policy from role")
safe(lambda: iam.delete_role(RoleName=ROLE_NAME),
     "Delete IAM role")

# 9. OTEL cleanup
provider.shutdown()

print(f"\n{'='*60}")
if errors:
    print(f"⚠️  {len(errors)} resource(s) may need manual cleanup")
else:
    print(f"✅ All resources deleted")

---
# Knowledge Check

## Concepts

1. **What is the difference between a trace and a span?**
   A trace is the end-to-end journey of a session. A span is one unit of work within that
   journey (e.g. one tool call). A trace contains many spans.

2. **Why is AI agent observability harder than traditional app observability?**
   Because the agent's tool call sequence is non-deterministic — the model decides which
   tools to call at runtime. You cannot predict the span tree; you must capture it.

3. **What span attribute revealed the Dark Tuesday root cause?**
   `novapay.used_default_flags = True` — the risk scorer used the default value of 0
   for sender_flags instead of the actual value of 3.

4. **What is the difference between head-based and tail-based sampling?**
   Head-based: decide at session start whether to trace (simpler, cheaper). Tail-based:
   trace everything, but only export traces that contain errors (catches more issues,
   costs more compute).

## Architecture

5. **Where do gateway spans go?**
   To CloudWatch Transaction Search via the log delivery pipeline configured with
   `put_delivery_source` / `put_delivery_destination` / `create_delivery`.

6. **How do you correlate spans across agent, gateway, and tools?**
   Using OpenTelemetry baggage with `session.id`, and the
   `X-Amzn-Bedrock-AgentCore-Runtime-Session-Id` header.

7. **What is ADOT and why does AgentCore use it?**
   AWS Distro for OpenTelemetry — the AWS-optimised OTEL distribution. It provides
   automatic instrumentation for Strands agents and exports to CloudWatch without
   custom shims.

## Operations

8. **What sampling rate does this lab recommend for NovaPay?**
   10% baseline + 100% for transactions over $50,000.

9. **What is the estimated monthly cost of observability for 12K sessions/day?**
   ~$5/month (CloudWatch Logs + Metrics + Alarms + X-Ray).

10. **Describe the 6-step incident response workflow.**
    Alarm → identify session → pull trace → read span tree → find anomaly in
    attributes → fix and verify with same inputs.

## Summary

| What you built | Why it matters |
|---|---|
| OpenTelemetry setup with InMemorySpanExporter | Captures spans locally for immediate analysis |
| Instrumented NovaPay tools with custom spans | Every tool call is traceable with structured attributes |
| Observable gateway with log and trace delivery | Distributed tracing across the full pipeline |
| CloudWatch alarms and custom metrics | Catch issues before customers (or compliance) do |
| Session correlation with OTEL baggage | Debug across agent boundaries with one session ID |
| Incident response workflow | 14-hour investigations become 15-minute span reads |

**Next:** Lab 10 brings evaluation — measuring whether your agents are actually making
good decisions, not just fast ones.